# X-ray 이물 검출 · 기준 모델 실험

경진대회 초기 단계의 YOLO 검출기 비교 노트북. 장비 표시 제거, 자료 분할, 학습, 판정 기준 비교와 오류 분석을 실행한다. 후속 CNN 박스 보정의 최종 성적과 비교표는 [프로젝트 설명](README.md)에 별도로 정리했다.

데이터 폴더와 이 노트북을 같은 위치에 둔다. 기본 모드는 소량으로 실행하는 smoke이며, 전체 학습은 XRAY_RUN_MODE 환경변수를 full로 설정한다. 전체 학습은 CUDA GPU를 전제로 한다.


## 1. 도구 불러오기
이미지를 읽고 숫자를 계산하고 모델을 학습하는 도구를 불러옵니다. 이 단계는 아직 학습이 아닙니다.

In [ ]:
from pathlib import Path
from collections import defaultdict
import os, json, hashlib, shutil, csv, io, zipfile, time, sys, platform, re, gc, html, base64
from datetime import datetime
import importlib.metadata as package_metadata
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import cv2, torch, yaml
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
assert (ROOT/'4. X-ray 검사장비 AI 데이터셋').is_dir(), '데이터 폴더와 baseline.ipynb가 있는 디렉터리에서 실행하세요.'
os.environ['YOLO_CONFIG_DIR'] = str(ROOT)
from ultralytics import YOLO, settings
settings.update({'sync': False, 'wandb': False, 'mlflow': False, 'comet': False})
plt.rcParams.update({'font.family':'Malgun Gothic','axes.unicode_minus':False})
torch.set_num_threads(4)

## 2. 실험 계획 고정하기
시험 답을 보기 전에 비교할 조건을 정하는 단계입니다.

|조건|뜻|
|---|---|
|n640_telea|작은 모델, 640 입력, Telea로 표시 제거|
|n960_telea|같은 모델에서 입력 크기만 960으로 변경|
|s640_telea|640 입력에서 더 큰 모델로 변경|
|n640_ns|기본 조건에서 표시 제거 방식만 NS로 변경|

한꺼번에 모든 것을 바꾸지 않아 어떤 변경이 도움이 됐는지 비교하기 쉽습니다. 시드 42·142·242는 학습 시작의 우연한 차이에 결과가 얼마나 흔들리는지 확인하는 반복입니다. 대표 모델의 시드는 **42로 미리 고정**합니다.

`recall_target=0.95`는 정답 100개 중 적어도 95개를 찾는 것을 목표로 한 실험 설정이며, 대회가 정한 합격선이나 안전 보증은 아닙니다. 목표를 달성하지 못하면 결과에 그대로 기록합니다.

처음에는 아래 기본 설정을 그대로 사용하세요. `external_root`는 별도로 확보한 외부 평가 자료가 있을 때만 지정합니다.

In [ ]:
# 기본 smoke는 실행 경로 확인용, full은 기준 모델 비교 학습.
MODE = os.environ.get('XRAY_RUN_MODE', 'smoke')
assert MODE in ('full', 'smoke')
CONFIG = {
    'mode': MODE, 'version':'submission-v1.0', 'split_seed':20260923,
    'seeds':[42,142,242], 'outer_folds':3, 'epochs':120, 'patience':20, 'nominal_batch_size':64,
    'recall_target':.95, 'iou_match':.5, 'nms_iou':.5, 'prediction_floor':.001,
    'final_calibration_fraction':.20, 'inner_tune_fraction':.25,
    'historical_test_start':'20200905', 'allow_cpu':False,
    'resume_interrupted':True, 'external_root':None,
    'candidates':[
        {'name':'n640_telea','weights':'yolo11n.pt','imgsz':640,'batch':8,'method':'telea'},
        {'name':'n960_telea','weights':'yolo11n.pt','imgsz':960,'batch':4,'method':'telea'},
        {'name':'s640_telea','weights':'yolo11s.pt','imgsz':640,'batch':4,'method':'telea'},
        {'name':'n640_ns','weights':'yolo11n.pt','imgsz':640,'batch':8,'method':'ns'},
    ],
}
if MODE == 'smoke':
    CONFIG.update(seeds=[42],epochs=2,patience=2,nominal_batch_size=4)
    CONFIG['candidates']=[dict(CONFIG['candidates'][0],imgsz=320,batch=4)]
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu' and not CONFIG['allow_cpu']:
    raise RuntimeError('CUDA GPU를 확인하세요. CPU 실행을 의도하면 allow_cpu=True로 바꾸세요.')
print('실행 모드:',MODE,' / GPU:',torch.cuda.get_device_name(0) if DEVICE==0 else 'CPU')
print('예정 학습 수:',CONFIG['outer_folds']*len(CONFIG['candidates'])*len(CONFIG['seeds'])+len(CONFIG['seeds']))

## 3. 파일과 정답을 읽는 공통 기능
TXT의 좌표가 정상 범위인지 확인하고, 파일 지문(SHA-256)으로 중복·변경을 확인합니다. IoU는 **정답 사각형과 예측 사각형이 얼마나 겹치는지** 나타내는 값입니다. 많이 겹칠수록 1에 가까워집니다.

In [ ]:
def digest(x): return hashlib.sha256(x).hexdigest()
def file_hash(p): return digest(Path(p).read_bytes())
def serial(obj):
    if isinstance(obj, (np.integer,np.floating,np.bool_)): return obj.item()
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, Path): return str(obj)
    raise TypeError(type(obj).__name__)
def dump(path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=serial),encoding='utf-8')
    tmp.replace(path)
def load(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def save_csv(df,path): df.to_csv(path,index=False,encoding='utf-8-sig')
def label_array(path):
    lines=Path(path).read_text(encoding='utf-8-sig').splitlines()
    a=np.array([list(map(float,l.split())) for l in lines if l.strip()],float).reshape(-1,5)
    if len(a):
        assert np.isfinite(a).all() and (a[:,0]==0).all(), path
        assert (a[:,3:]>0).all(),path
        assert (a[:,1:3]-a[:,3:]/2 >= -1e-7).all() and (a[:,1:3]+a[:,3:]/2<=1+1e-7).all(),path
    return a
def label_key(a):return sorted(tuple(np.round(r,7)) for r in a)
def xyxy(a,w,h):
    centers=a[:,1:3]*[w,h]; wh=a[:,3:]*[w,h]
    return np.column_stack((centers-wh/2,centers+wh/2))
def ious(a,b):
    a=np.asarray(a,float).reshape(-1,4);b=np.asarray(b,float).reshape(-1,4)
    inter=np.maximum(np.minimum(a[:,None,2:],b[None,:,2:])-np.maximum(a[:,None,:2],b[None,:,:2]),0).prod(2)
    aa=np.maximum(a[:,2:]-a[:,:2],0).prod(1);bb=np.maximum(b[:,2:]-b[:,:2],0).prod(1)
    return inter/np.maximum(aa[:,None]+bb[None,:]-inter,1e-12)

## 4. 사용할 자료 확정하기
같은 이미지의 복사본을 새 자료로 세지 않습니다. 주 라벨 500장 중 다른 TXT 판본과 충돌하는 3장은 임의로 정답을 고르지 않고 보류합니다. 따라서 기본 분석 대상은 497장·이물질 1,140개입니다. 라벨이 없다는 이유로 정상이라고 판단하지 않습니다.

파일명에서 날짜를 읽고 경로에서 장비 번호를 확인합니다. 이 정보는 자료를 나누고 결과를 분석하는 데 사용하며 모델의 입력 특징으로 넣지 않습니다.

In [ ]:
def inventory():
    rawroot=ROOT/'4. X-ray 검사장비 AI 데이터셋'
    labels=next(rawroot.rglob('라벨링 6종 세트'))/'labels'
    bmps=defaultdict(list)
    for p in rawroot.rglob('*.bmp'):bmps[p.stem].append(p)
    conflicts=[];extras=[]
    for p in rawroot.rglob('*.txt'):
        if p.parent==labels or not re.match(r'^\d{3}_\d{8}_',p.stem):continue
        q=labels/p.name
        if not q.exists():extras.append(str(p.relative_to(ROOT)));continue
        if label_key(label_array(p))!=label_key(label_array(q)):
            conflicts.append({'image_id':p.stem,'main':str(q.relative_to(ROOT)),'alternate':str(p.relative_to(ROOT))})
    exclude={r['image_id'] for r in conflicts}; rows=[]
    for label in sorted(labels.glob('*.txt')):
        if label.stem in exclude:continue
        paths=bmps.get(label.stem,[]);assert paths,f'BMP 없음: {label.name}'
        hashes={digest(np.asarray(Image.open(p).convert('RGB')).tobytes()) for p in paths}
        assert len(hashes)==1,f'동일 이름의 다른 영상: {label.stem}'
        raw=sorted(paths,key=lambda p:('호기' not in str(p),str(p)))[0]
        a=np.asarray(Image.open(raw).convert('RGB'));h,w=a.shape[:2]
        machine=re.search(r'([123])호기',str(raw));assert machine,raw
        arr=label_array(label)
        rows.append(dict(image_id=label.stem,raw=str(raw.relative_to(ROOT)),label=str(label.relative_to(ROOT)),
            date=label.stem.split('_')[1],machine=machine.group(1),width=w,height=h,n_objects=len(arr),
            pixel_sha256=next(iter(hashes)),raw_sha256=file_hash(raw),label_sha256=file_hash(label)))
    m=pd.DataFrame(rows).sort_values(['date','image_id']).reset_index(drop=True)
    assert not m.pixel_sha256.duplicated().any(),'중복 영상이 있어 먼저 그룹을 확인해야 합니다.'
    m['group']=m.date
    m['count_bin']=np.where(m.n_objects>1,'multi',np.where(m.n_objects==1,'single','negative'))
    m['stratum']=m.machine+'_'+m.count_bin
    m['cohort']=np.where(m.date>=CONFIG['historical_test_start'],'historical_seen_test','development')
    audit={'main_txt':len(list(labels.glob('*.txt'))),'eligible_images':len(m),'objects':int(m.n_objects.sum()),
        'quarantined_images':len(exclude),'conflicts':conflicts,'extra_tutorial_txt':extras,
        'confirmed_negatives':int((m.n_objects==0).sum()),'bmp_files':sum(map(len,bmps.values())),
        'policy':'주 라벨 묶음 사용; TXT 판본 충돌은 보류; 라벨 없는 영상은 정상으로 취급하지 않음'}
    return m,audit

## 5. 색상 표시 제거와 데이터 탐색(EDA)
색이 있는 픽셀을 찾아 주변까지 조금 넓힌 뒤, 주변 밝기를 이용해 메웁니다. **이 함수에는 TXT 위치를 주지 않습니다.** 따라서 새 이미지에도 같은 방법을 적용할 수 있습니다.

이물질 크기(픽셀), 주변과의 밝기 차이, 제품 가장자리까지의 추정 거리, 박스의 가로세로 비율 등을 기록합니다. 이 숫자들은 나중에 어떤 조건에서 놓쳤는지 설명하는 용도입니다. 물질의 실제 밀도나 정확한 모양을 측정한 값은 아닙니다.

In [ ]:
def clean_rgb(rgb,method='telea',dilation=3):
    # 정답 좌표를 받지 않습니다. 추론에서도 동일한 함수가 실행됩니다.
    color=(np.ptp(rgb.astype(np.int16),axis=2)>20).astype(np.uint8)*255
    mask=cv2.dilate(color,np.ones((dilation,dilation),np.uint8))
    gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)
    flag=cv2.INPAINT_TELEA if method=='telea' else cv2.INPAINT_NS
    cleaned=cv2.inpaint(gray,mask,3,flag) if mask.any() else gray.copy()
    return cleaned,mask
def product_mask(gray):
    _,mask=cv2.threshold(cv2.GaussianBlur(gray,(7,7),0),0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((9,9),np.uint8))
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    out=np.zeros_like(mask)
    if contours:cv2.drawContours(out,[max(contours,key=cv2.contourArea)],-1,255,-1)
    return out
def prepare(m,work):
    methods=sorted({c['method'] for c in CONFIG['candidates']}|{'telea','ns'})
    rows=[]; prep=[]; truth={}
    for method in methods:
        for d in ('images','labels'):(work/'prepared'/method/d).mkdir(parents=True,exist_ok=True)
    for r in m.itertuples():
        rgb=np.asarray(Image.open(ROOT/r.raw).convert('RGB')); a=label_array(ROOT/r.label)
        boxes=xyxy(a,r.width,r.height);truth[r.image_id]=boxes
        for method in methods:
            gray,mask=clean_rgb(rgb,method)
            p=work/'prepared'/method/'images'/(r.image_id+'.png')
            Image.fromarray(gray).convert('RGB').save(p)
            shutil.copyfile(ROOT/r.label,work/'prepared'/method/'labels'/(r.image_id+'.txt'))
        gray,mask=clean_rgb(rgb,'telea');region=product_mask(gray)
        distance=cv2.distanceTransform(region,cv2.DIST_L2,5)
        prep.append(dict(image_id=r.image_id,colored_pixels=int((np.ptp(rgb.astype(np.int16),axis=2)>20).sum()),
                         inpaint_fraction=float((mask>0).mean())))
        occupied=np.zeros(gray.shape,bool)
        for b in boxes:
            x1,y1=np.floor(b[:2]).astype(int);x2,y2=np.ceil(b[2:]).astype(int);occupied[y1:y2,x1:x2]=True
        for k,b in enumerate(boxes):
            x1,y1=np.floor(b[:2]).astype(int);x2,y2=np.ceil(b[2:]).astype(int)
            bw,bh=b[2:]-b[:2];cx,cy=((b[:2]+b[2:])/2).astype(int)
            roi=np.zeros(gray.shape,bool);roi[y1:y2,x1:x2]=True
            ring=cv2.dilate(roi.astype(np.uint8),np.ones((13,13),np.uint8)).astype(bool)&~occupied&(mask==0)
            vals=gray[roi&(mask==0)].astype(float);bg=gray[ring].astype(float)
            cnr=abs(vals.mean()-bg.mean())/max(bg.std(),1.) if len(vals)>=3 and len(bg)>=10 else np.nan
            rows.append(dict(image_id=r.image_id,object_id=k,date=r.date,machine=r.machine,count_bin=r.count_bin,
                x1=b[0],y1=b[1],x2=b[2],y2=b[3],min_side_px=min(bw,bh),area_px=bw*bh,
                aspect_ratio=max(bw,bh)/min(bw,bh),contrast_cnr=cnr,
                edge_distance_px=float(distance[min(cy,r.height-1),min(cx,r.width-1)]),
                inpaint_overlap=float((mask[roi]>0).mean())))
    save_csv(pd.DataFrame(rows),work/'object_features.csv');save_csv(pd.DataFrame(prep),work/'preprocess_audit.csv')
    return truth,pd.DataFrame(rows)

## 6. 공부 자료와 시험 자료 분리하기

1. 기존 시험 75장은 따로 보관합니다.
2. 개발 자료에서 최종 모델의 점수 기준을 정할 보정 자료를 날짜 단위로 따로 확보합니다.
3. 나머지 자료를 3개 바깥 평가 묶음으로 나눕니다. 한 묶음이 시험이면 다른 자료에서만 학습합니다.
4. 각 반복의 학습 부분도 내부 학습/내부 검증으로 나눕니다. 내부 검증으로 모델과 기준을 고르고, 바깥 시험에서는 바꾸지 않습니다.

**같은 날짜·같은 픽셀 영상이 경계를 넘으면 코드가 멈춥니다.** 가능한 범위에서 장비·이물질 개수의 분포도 고려합니다. 자료 수가 적어서 모든 조건을 완벽하게 맞출 수는 없습니다. 실제 제품 ID가 없어 날짜가 다른 동일 제품까지 완전히 배제했다고 할 수는 없습니다.

In [ ]:
def split_cost(full,parts,fractions):
    # 성능을 전혀 보지 않고, 영상 수와 장비/개수 분포만으로 고릅니다.
    labels=sorted(full.stratum.unique()); target=full.stratum.value_counts(normalize=True).reindex(labels,fill_value=0)
    cost=0.
    for p,frac in zip(parts,fractions):
        cost+=4*abs(len(p)/len(full)-frac)
        cost+=float((p.stratum.value_counts(normalize=True).reindex(labels,fill_value=0)-target).abs().sum())
        cost+=5*len(set(full.machine)-set(p.machine))
        cost+=8*len(set(full.count_bin)-set(p.count_bin))
    return cost
def group_holdout(df,fraction,seed):
    splitter=GroupShuffleSplit(n_splits=512,test_size=fraction,random_state=seed)
    options=[]
    for a,b in splitter.split(df,groups=df.group):
        fit,tune=df.iloc[a].copy(),df.iloc[b].copy()
        if fit.group.nunique()<3 or tune.group.nunique()<2:continue
        options.append((split_cost(df,[fit,tune],[1-fraction,fraction]),a,b))
    assert options,'날짜 그룹이 너무 적어 내부 분리를 만들 수 없습니다.'
    _,a,b=min(options,key=lambda x:x[0]);return df.iloc[a].copy(),df.iloc[b].copy()
def outer_splits(core):
    options=[]
    for s in range(CONFIG['split_seed'],CONFIG['split_seed']+64):
        split=StratifiedGroupKFold(n_splits=CONFIG['outer_folds'],shuffle=True,random_state=s)
        parts=list(split.split(core,core.stratum,core.group))
        cost=split_cost(core,[core.iloc[b] for a,b in parts],[1/CONFIG['outer_folds']]*CONFIG['outer_folds'])
        options.append((cost,parts))
    return min(options,key=lambda x:x[0])[1]
def assert_disjoint(*parts):
    for i,a in enumerate(parts):
        for b in parts[i+1:]:
            assert not(set(a.group)&set(b.group)),'날짜 중복'
            assert not(set(a.pixel_sha256)&set(b.pixel_sha256)),'원본 픽셀 중복'
def tiny_sample(df):
    # smoke만 사용. 실제 통계용 자료는 아닙니다.
    return df.groupby(['machine','count_bin'],group_keys=False).head(3).copy()
def plan_splits(m,work):
    dev=m[m.cohort=='development'].copy();historical=m[m.cohort=='historical_seen_test'].copy()
    core,cal=group_holdout(dev,CONFIG['final_calibration_fraction'],CONFIG['split_seed'])
    assert_disjoint(core,cal,historical)
    plans=[];audit=[]
    for fold,(a,b) in enumerate(outer_splits(core)):
        outertrain,outertest=core.iloc[a].copy(),core.iloc[b].copy()
        fit,tune=group_holdout(outertrain,CONFIG['inner_tune_fraction'],CONFIG['split_seed']+100+fold)
        assert_disjoint(fit,tune,outertest,cal,historical)
        if MODE=='smoke':fit,tune,outertest=map(tiny_sample,(fit,tune,outertest))
        plans.append((fit,tune,outertest))
        for role,df in [('fit',fit),('inner_tune',tune),('outer_evaluation',outertest)]:
            for r in df.itertuples():audit.append(dict(fold=fold,role=role,image_id=r.image_id,date=r.date,machine=r.machine,count_bin=r.count_bin))
    save_csv(pd.DataFrame(audit),work/'nested_split_assignments.csv')
    final=m[['image_id','date','machine','n_objects']].copy()
    final['role']=final.image_id.map({**dict.fromkeys(core.image_id,'final_fit'),**dict.fromkeys(cal.image_id,'final_calibration'),**dict.fromkeys(historical.image_id,'historical_seen_test')})
    save_csv(final,work/'final_split_assignments.csv')
    support=pd.DataFrame(audit).groupby(['fold','role','machine','count_bin']).size().rename('images').reset_index()
    save_csv(support,work/'split_support.csv')
    if MODE=='full':
        ids=pd.concat([p[2] for p in plans]).image_id
        assert not ids.duplicated().any() and set(ids)==set(core.image_id)
    if MODE=='smoke':core,cal,historical=map(tiny_sample,(core,cal,historical))
    return core,cal,historical,plans

## 7. 실제 학습과 재시작
공개 사전학습 모델에서 출발해 우리 자료로 가중치를 바꿉니다. 새로 처음부터 모든 것을 배우는 것보다 적은 자료에서도 시작하기 좋습니다. 완료한 모델은 가중치 지문과 학습 기록을 함께 저장합니다.

여기서는 학습 함수를 정의합니다. **마지막 실행 셀에서 실제 학습이 시작됩니다.** GPU 메모리 오류가 나면 설정의 `batch`를 줄이고 다시 실행하세요. 변경된 설정은 새로운 실험으로 기록됩니다.

In [ ]:
def get_weights(name):
    p=ROOT/'models'/name
    if not p.exists():
        import urllib.request
        p.parent.mkdir(exist_ok=True)
        temporary=p.with_suffix('.download')
        urllib.request.urlretrieve('https://github.com/ultralytics/assets/releases/download/v8.3.0/'+name,temporary)
        temporary.replace(p)
    return p
def write_data_yaml(work,tag,method,fit,tune,test=None):
    directory=work/'splits'/tag;directory.mkdir(parents=True,exist_ok=True)
    ds={}
    for split,df in [('train',fit),('val',tune),('test',test)]:
        if df is None:continue
        paths=[str((work/'prepared'/method/'images'/(x+'.png')).resolve()) for x in df.image_id]
        txt=directory/(split+'.txt');txt.write_text('\n'.join(paths)+'\n',encoding='utf-8');ds[split]=str(txt.resolve())
    config={'path':str(work.resolve()),**ds,'names':{0:'foreign_object'}}
    p=directory/'data.yaml';p.write_text(yaml.safe_dump(config,allow_unicode=True),encoding='utf-8');return p
def fit_model(work,tag,candidate,seed,fit,tune,historical=None):
    initial=get_weights(candidate['weights'])
    args=dict(epochs=CONFIG['epochs'],patience=CONFIG['patience'],imgsz=candidate['imgsz'],batch=candidate['batch'],
        seed=seed,workers=0,device=DEVICE,optimizer='AdamW',lr0=.001,cos_lr=True,amp=False,nbs=CONFIG['nominal_batch_size'],
        deterministic=True,cache=False,hsv_h=0.,hsv_s=0.,hsv_v=0.,degrees=5.,translate=.05,
        scale=.15,fliplr=.5,flipud=0.,mosaic=0.,mixup=0.,plots=True,verbose=False,save=True,exist_ok=True)
    signature=digest(json.dumps({'candidate':candidate,'args':args,'fit':sorted(fit.image_id),
        'tune':sorted(tune.image_id),'initial':file_hash(initial),'contract':load(work/'contract.json')['id']},sort_keys=True).encode())
    run=work/'runs'/tag;done=run/'completed.json';checkpoint=run/'weights'/'best.pt'
    if done.exists():
        record=load(done)
        assert record['signature']==signature and checkpoint.exists() and file_hash(checkpoint)==record['checkpoint_sha256'],'학습 캐시가 변경되었습니다.'
        print('재사용:',tag,flush=True);return record
    run.mkdir(parents=True,exist_ok=True)
    data=write_data_yaml(work,tag,candidate['method'],fit,tune,historical)
    dump(run/'requested.json',{'signature':signature,'args':args,'fit_ids':fit.image_id.tolist(),'tune_ids':tune.image_id.tolist()})
    started=time.time();last=run/'weights'/'last.pt';resume=False
    if last.exists() and CONFIG['resume_interrupted']:
        saved=torch.load(last,map_location='cpu',weights_only=False)
        resume=saved.get('epoch',-1)>=0 and saved.get('optimizer') is not None
        del saved
    print('학습:',tag,'fit',len(fit),'tune',len(tune),'resume',resume,flush=True)
    model=YOLO(str(last if resume else initial))
    if resume:model.train(resume=True,device=DEVICE)
    else:model.train(data=str(data),project=str(work/'runs'),name=tag,**args)
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    assert checkpoint.exists(),'학습 모델 저장 실패'
    initial_state=YOLO(str(initial)).model.state_dict();trained_state=YOLO(str(checkpoint)).model.state_dict()
    changed_weights=sum(not torch.equal(initial_state[k],trained_state[k]) for k in initial_state
        if k in trained_state and initial_state[k].shape==trained_state[k].shape and k.endswith('weight'))
    assert changed_weights>0,'사전학습 모델과 공통 가중치가 같아 실제 업데이트 여부를 확인해야 합니다.'
    del initial_state,trained_state
    logs=pd.read_csv(run/'results.csv');column='metrics/mAP50-95(B)'
    j=logs.index[logs[column]==logs[column].max()][-1]  # Ultralytics는 최고 점수 동률에서도 best.pt를 갱신합니다.
    record={'signature':signature,'checkpoint':str(checkpoint),'checkpoint_sha256':file_hash(checkpoint),
        'candidate':candidate,'seed':seed,'epochs_run':len(logs),'best_epoch':int(logs.loc[j,'epoch']),
        'training_log_seconds':float(logs.time.iloc[-1]),'this_execution_seconds':time.time()-started,
        'data_yaml':str(data),'tag':tag,'best_tune_map50_95':float(logs.loc[j,column]),'changed_shared_weight_tensors':changed_weights}
    dump(done,record);return record

## 8. 채점 방법과 판정 기준

- **TP**: 이물질을 맞게 찾은 개수입니다. 위치도 IoU 0.5 이상 겹쳐야 합니다.
- **FN**: 정답은 있는데 찾지 못한 개수입니다. 위치를 부정확하게 잡아도 해당합니다.
- **FP**: 정답과 맞지 않는 예측입니다. 같은 물체에 박스를 중복으로 그리면 추가 정답으로 세지 않습니다.
- **재현율**: 전체 정답 중 찾은 비율입니다.
- **정밀도**: 모델이 찾았다고 한 것 중 맞은 비율입니다.
- **F2**: 정밀도와 재현율을 합치되, 놓치지 않는 쪽에 더 무게를 둔 지표입니다.

점수 기준은 내부 검증/최종 보정 자료에서만 정합니다. 시험 점수가 좋아지도록 기준을 뒤에서 바꾸지 않습니다. 작은 물체·낮은 대비 등의 경계도 학습 자료에서 정합니다. 조건별 비교는 관련성을 보여주며 원인을 증명하지는 않습니다.

In [ ]:
def predict(work,record,df,tag):
    cache=work/'predictions'/(tag+'.json');signature=digest(json.dumps({'model':record['checkpoint_sha256'],
        'ids':df.image_id.tolist(),'pixels':df.pixel_sha256.tolist(),'candidate':record['candidate'],'floor':CONFIG['prediction_floor'],'nms':CONFIG['nms_iou']},sort_keys=True).encode())
    if cache.exists():
        saved=load(cache)
        if saved['signature']==signature:return saved['predictions']
    model=YOLO(record['checkpoint']);c=record['candidate']
    paths=[str(work/'prepared'/c['method']/'images'/(x+'.png')) for x in df.image_id]
    out={}
    for r,p in zip(df.itertuples(),model.predict(paths,imgsz=c['imgsz'],batch=c['batch'],conf=CONFIG['prediction_floor'],
        iou=CONFIG['nms_iou'],max_det=100,device=DEVICE,verbose=False,stream=True)):
        assert Path(p.path).stem==r.image_id
        out[r.image_id]=np.column_stack((p.boxes.xyxy.cpu().numpy(),p.boxes.conf.cpu().numpy())).tolist()
    assert set(out)==set(df.image_id)
    dump(cache,{'signature':signature,'predictions':out});del model
    return out
def match(gt,pred,iou=.5):
    p=np.asarray(pred,float).reshape(-1,5);p=p[np.argsort(-p[:,4])]
    assigned=np.zeros(len(gt),bool);scores=np.zeros(len(gt));fp=[];overlap=ious(p[:,:4],gt)
    for j,b in enumerate(p):
        options=np.flatnonzero((overlap[j]>=iou)&~assigned)
        if len(options):
            k=options[np.argmax(overlap[j,options])];assigned[k]=True;scores[k]=b[4]
        else:fp.append(float(b[4]))
    return scores,np.asarray(fp),p,overlap
def score(pred,df,truth,threshold,iou=.5):
    image_rows=[];obj_rows=[]
    for r in df.itertuples():
        g=truth[r.image_id];s,f,p,overlap=match(g,pred[r.image_id],iou)
        hit=s>=threshold;tp=int(hit.sum());fp=int((f>=threshold).sum());fn=len(g)-tp
        image_rows.append(dict(image_id=r.image_id,date=r.date,machine=r.machine,n_objects=len(g),tp=tp,fp=fp,fn=fn,
            all_found=bool(hit.all()) if len(g) else None,any_found=bool(hit.any()),image_alert=bool((p[:,4]>=threshold).any())))
        above=p[:,4]>=threshold
        for k,h in enumerate(hit):
            best=float(overlap[above,k].max()) if above.any() else 0.
            obj_rows.append(dict(image_id=r.image_id,object_id=k,detected=bool(h),score=float(s[k]),best_iou=best,
                failure_type='pass' if h else 'localization' if best>=.25 else 'no_matching_detection'))
    images=pd.DataFrame(image_rows);objects=pd.DataFrame(obj_rows,columns=['image_id','object_id','detected','score','best_iou','failure_type'])
    tp,fp,fn=map(int,images[['tp','fp','fn']].sum());negative=images[images.n_objects==0]
    result={'threshold':float(threshold),'iou':iou,'images':len(df),'objects':tp+fn,'tp':tp,'fp':fp,'fn':fn,
        'precision':tp/max(tp+fp,1),'recall':tp/max(tp+fn,1),'f1':2*tp/max(2*tp+fp+fn,1),
        'f2':5*tp/max(5*tp+fp+4*fn,1),'fp_per_image':fp/max(len(df),1),
        'all_found_images':int(images[images.n_objects>0].all_found.sum()),
        'normal_images':len(negative),'normal_false_alarm_rate':float(negative.image_alert.mean()) if len(negative) else None}
    return result,images,objects
def threshold_policy(pred,df,truth):
    matched=[];false=[];total=0
    for r in df.itertuples():
        s,f,_,_=match(truth[r.image_id],pred[r.image_id],CONFIG['iou_match'])
        matched.extend(s);false.extend(f);total+=len(s)
    matched=np.asarray(matched);false=np.asarray(false);rows=[]
    for t in np.unique(np.r_[CONFIG['prediction_floor'],.005,np.arange(.01,.951,.01)]):
        tp=int((matched>=t).sum());fp=int((false>=t).sum());fn=total-tp
        rows.append(dict(threshold=float(t),tp=tp,fp=fp,fn=fn,precision=tp/max(tp+fp,1),recall=tp/max(total,1),
            f1=2*tp/max(2*tp+fp+fn,1),f2=5*tp/max(5*tp+fp+4*fn,1)))
    grid=pd.DataFrame(rows);feasible=grid[grid.recall>=CONFIG['recall_target']]
    best=(feasible.sort_values(['precision','threshold'],ascending=False).iloc[0] if len(feasible) else
          grid.sort_values(['recall','precision','threshold'],ascending=False).iloc[0])
    low=float(best.threshold);high=max(low,float(grid.sort_values(['f1','threshold'],ascending=False).iloc[0].threshold))
    policy={'low':low,'high':high,'target_met':bool(len(feasible)),'target':CONFIG['recall_target'],
            'automatic_release':False,'score_is_probability':False,'empty_middle_band':low==high}
    return policy,grid
def choose_candidate(records):
    d=pd.DataFrame(records)
    summary=d.groupby('candidate',sort=False).agg(mean_f2=('f2','mean'),mean_precision=('precision','mean'),
        worst_recall=('recall','min'),recall_std=('recall','std'),goal_fraction=('target_met','mean'),runs=('seed','size')).reset_index()
    good=summary[summary.goal_fraction==1]
    chosen=(good.sort_values(['mean_f2','mean_precision'],ascending=False,kind='stable').iloc[0] if len(good) else
            summary.sort_values(['worst_recall','mean_f2','mean_precision'],ascending=False,kind='stable').iloc[0])
    return str(chosen.candidate),summary
def conditions(obj,features,fit_ids):
    d=features.merge(obj,on=['image_id','object_id'],validate='one_to_one')
    train=features[features.image_id.isin(fit_ids)]
    cut={c:float(train[c].quantile(.25)) for c in ('min_side_px','contrast_cnr','edge_distance_px')}
    groups={c:pd.Series(np.where(d[c].isna(),'unknown',np.where(d[c]<=cut[c],'train_Q25_or_less','above_train_Q25')),index=d.index) for c in cut}
    groups.update(machine=d.machine.astype(str),count_bin=d.count_bin,
        aspect_ratio=pd.Series(np.where(d.aspect_ratio>=1.5,'elongated_1.5_plus','below_1.5'),index=d.index),
        inpaint_overlap=pd.Series(np.where(d.inpaint_overlap>.25,'above_25pct','up_to_25pct'),index=d.index))
    rows=[]
    for condition,series in groups.items():
        expected=['above_25pct','up_to_25pct'] if condition=='inpaint_overlap' else sorted(series.unique())
        for group in expected:
            sub=d[series==group];n=len(sub);hit=int(sub.detected.sum())
            rows.append(dict(condition=condition,group=group,objects=n,detected=hit,missed=n-hit,
                recall=hit/n if n else np.nan,images=sub.image_id.nunique(),cutoff=cut.get(condition)))
    return pd.DataFrame(rows),d
def cluster_interval(images):
    g=images.groupby('date')[['tp','fn']].sum();rng=np.random.default_rng(42);values=[]
    if len(g)<2:return {'dates':len(g),'interval':None}
    for _ in range(1000):
        a=g.iloc[rng.integers(0,len(g),len(g))].sum();values.append(float(a.tp/max(a.tp+a.fn,1)))
    return {'dates':len(g),'interval':np.quantile(values,[.025,.975]).tolist(),'note':'날짜 묶음 재표집의 경험적 구간; 표본 부족과 날짜 간 의존 가능성은 남음'}

## 9. 채점 코드 자체를 확인하기
중복 예측, 정답이 없는 영상, 검출이 없는 상황, 표시 제거 등을 작은 정답이 확실한 사례로 검증합니다. 모델 점수가 높더라도 채점 코드가 틀리면 의미가 없으므로 먼저 확인합니다.

In [ ]:
def protocol_tests():
    g=np.array([[0,0,10,10],[20,20,30,30]])
    s,f,_,_=match(g,[[0,0,10,10,.9],[0,0,10,10,.8],[80,80,90,90,.7]])
    assert (s>0).tolist()==[True,False] and len(f)==2
    assert len(match(np.empty((0,4)),[[0,0,10,10,.9]])[1])==1
    assert not match(g,[])[0].any()
    plain=np.full((40,40,3),128,np.uint8)
    assert np.array_equal(clean_rgb(plain)[0],plain[:,:,0])
    marked=plain.copy();marked[10:25,10]=[255,0,0]
    out,mask=clean_rgb(marked);assert mask.any() and abs(out.astype(float)-128).max()<5
    # 낮은 점수를 제거할 때 매칭 결과가 고득점 우선 규칙과 일치하는지 확인합니다.
    pred=[[0,0,10,10,.9],[20,20,30,30,.4]]
    a=match(g,pred)[0]>=.5;b=match(g,[p for p in pred if p[4]>=.5])[0]>=.5
    assert np.array_equal(a,b)
    print('평가/빈 정답/중복 예측/전처리 기본 검증 통과')

## 10. 모델이 엉뚱한 단서를 외웠는지 점검하기
① 영상 내용을 보지 않고 흔한 위치에 박스를 그리는 단순 기준과 비교합니다.

② 색상 표시 제거 방법과 제거 범위를 바꾸었을 때 결과가 얼마나 달라지는지 확인합니다.

③ 실제 이물질과 겹치지 않는 곳에 가짜 표시를 추가한 후 지워봅니다. 그 흔적 때문에 새 경보가 생기는지 확인합니다. 이는 정상 제품 오경보율 측정과는 다른 보조 시험입니다.

In [ ]:
def position_baseline(fit,evaluation,truth):
    # 영상을 보지 않고 학습 TXT의 위치 중앙값만 쓰는 비교 기준입니다.
    # 이미지 크기는 알 수 있지만 시험 정답 개수는 사용하지 않습니다.
    samples=defaultdict(list)
    for r in fit.itertuples():
        samples[(r.width,r.height)].extend((truth[r.image_id]/[r.width,r.height,r.width,r.height]).tolist())
    all_boxes=[b for v in samples.values() for b in v];assert all_boxes
    out={}
    for r in evaluation.itertuples():
        b=np.median(samples.get((r.width,r.height),all_boxes),axis=0)*[r.width,r.height,r.width,r.height]
        out[r.image_id]=[b.tolist()+[1.0]]
    return out
def stress_tests(work,record,policy,cal,truth):
    model=YOLO(record['checkpoint']);c=record['candidate'];subset=cal.sort_values('image_id').head(48).copy()
    if MODE=='smoke':subset=subset.head(4)
    def batch_predict(arrays):
        out={}
        for r,p in zip(subset.itertuples(),model.predict(arrays,imgsz=c['imgsz'],batch=c['batch'],
            conf=CONFIG['prediction_floor'],iou=CONFIG['nms_iou'],max_det=100,device=DEVICE,verbose=False,stream=True)):
            out[r.image_id]=np.column_stack((p.boxes.xyxy.cpu().numpy(),p.boxes.conf.cpu().numpy())).tolist()
        return out
    sensitivity=[]
    for method,dilation in [(c['method'],3),('ns' if c['method']=='telea' else 'telea',3),(c['method'],5)]:
        arrays=[]
        for r in subset.itertuples():
            rgb=np.asarray(Image.open(ROOT/r.raw).convert('RGB'));gray,_=clean_rgb(rgb,method,dilation)
            arrays.append(cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR))
        pred=batch_predict(arrays)
        sensitivity.append(dict(method=method,dilation=dilation,**score(pred,subset,truth,policy['low'])[0]))
    save_csv(pd.DataFrame(sensitivity),work/'preprocessing_stress.csv')
    rng=np.random.default_rng(191);arrays=[];decoys={};base_arrays=[];examples=[]
    for r in subset.itertuples():
        rgb=np.asarray(Image.open(ROOT/r.raw).convert('RGB'));gray,mask=clean_rgb(rgb,c['method'])
        base_arrays.append(cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR))
        region=product_mask(gray);yy,xx=np.where(cv2.erode(region,np.ones((25,25),np.uint8))>0)
        b=None
        for _ in range(300):
            if not len(xx):break
            j=rng.integers(0,len(xx));x,y=int(xx[j]),int(yy[j]);candidate=[x-6,y-6,x+6,y+6]
            if mask[y-8:y+9,x-8:x+9].any():continue
            if len(truth[r.image_id]) and ious([candidate],truth[r.image_id]).max()>.001:continue
            b=candidate;break
        modified=rgb.copy()
        if b is not None:
            cv2.rectangle(modified,tuple(b[:2]),tuple(b[2:]),(255,0,0),1);decoys[r.image_id]=b
        changed,_=clean_rgb(modified,c['method']);arrays.append(cv2.cvtColor(changed,cv2.COLOR_GRAY2BGR))
        if b is not None and len(examples)<3:examples.append((gray,changed,b))
    before=batch_predict(base_arrays);after=batch_predict(arrays);decoy_rows=[]
    for image_id,b in decoys.items():
        flags={}
        for name,pred in [('before',before),('after',after)]:
            boxes=[p[:4] for p in pred[image_id] if p[4]>=policy['low']]
            flags[name]=bool(len(boxes) and ious([b],boxes).max()>=.25)
        decoy_rows.append(dict(image_id=image_id,**flags,new_alert=flags['after'] and not flags['before']))
    decoy=pd.DataFrame(decoy_rows,columns=['image_id','before','after','new_alert'])
    save_csv(decoy,work/'synthetic_marker_control.csv')
    result={'trials':len(decoy),'new_alerts':int(decoy.new_alert.sum()),
        'new_alert_rate':float(decoy.new_alert.mean()) if len(decoy) else None,
        'note':'TXT와 겹치지 않는 제품 내부에 12px 색상 사각형을 추가한 후 제거한 대조시험. 실제 정상 제품 오경보율 아님.'}
    dump(work/'artifact_diagnostic.json',result)
    if examples:
        fig,ax=plt.subplots(len(examples),2,figsize=(8,3*len(examples)),squeeze=False)
        for i,(base,changed,b) in enumerate(examples):
            for j,im in enumerate([base,changed]):
                ax[i,j].imshow(im,cmap='gray');ax[i,j].axis('off')
                ax[i,j].add_patch(plt.Rectangle(b[:2],b[2]-b[0],b[3]-b[1],fill=False,color='orange'))
            ax[i,0].set_title('원래 전처리본');ax[i,1].set_title('가짜 표시를 추가한 뒤 제거')
        fig.suptitle('주황 박스는 대조 위치 설명용이며 모델 입력에 없음')
        fig.tight_layout();fig.savefig(work/'figures'/'artifact_control.png',dpi=160);plt.close(fig)
    del model;return result

## 11. 새 외부 자료가 있을 때의 평가
기본값은 외부 자료가 없어 미실행입니다. 준비한 경우 `external_root` 폴더 안의 `metadata.csv`에 `image_file,label_file,group_id,machine` 열을 넣으세요. 이미지는 BMP/PNG 등, TXT는 동일 YOLO 형식입니다. **정상으로 확인된 영상은 빈 TXT를 명시적으로 준비해야 합니다.**

이 자료는 모델/점수 기준을 고정한 뒤 평가에만 씁니다. 기존 픽셀과 중복이면 중단합니다. 실제 제품 중복 여부와 정답의 신뢰성은 제공자가 확인해야 합니다.

In [ ]:
def external_data(work,manifest,truth):
    if not CONFIG['external_root']:return None
    root=Path(CONFIG['external_root']).resolve();metadata=pd.read_csv(root/'metadata.csv',dtype=str)
    assert {'image_file','label_file','group_id','machine'}<=set(metadata.columns)
    rows=[];known=set(manifest.pixel_sha256);seen=set()
    for i,r in enumerate(metadata.itertuples()):
        raw=root/r.image_file;label=root/r.label_file
        assert raw.is_file() and label.is_file(),'외부 정상 영상도 명시적 빈 TXT가 필요합니다.'
        rgb=np.asarray(Image.open(raw).convert('RGB'));h,w=rgb.shape[:2];pixel=digest(rgb.tobytes())
        assert pixel not in known and pixel not in seen,'기존 자료 또는 외부 자료 내 중복 영상'
        seen.add(pixel);a=label_array(label);image_id=f'external_{i:05d}'
        truth[image_id]=xyxy(a,w,h)
        for method in ('telea','ns'):
            gray,_=clean_rgb(rgb,method);Image.fromarray(gray).convert('RGB').save(work/'prepared'/method/'images'/(image_id+'.png'))
            shutil.copyfile(label,work/'prepared'/method/'labels'/(image_id+'.txt'))
        rows.append(dict(image_id=image_id,date=str(r.group_id),group=str(r.group_id),machine=str(r.machine),
            width=w,height=h,n_objects=len(a),pixel_sha256=pixel,raw_sha256=file_hash(raw),label_sha256=file_hash(label)))
    df=pd.DataFrame(rows);assert len(df)>0
    save_csv(df,work/'external_manifest.csv');return df

## 12. 실행 결과로 보고서 작성하기
자료 분리, 후보 비교, 시드별 성능, 미탐지 조건, 표시 제거 영향, 판정/재검사 제안을 표와 그림으로 자동 정리합니다. 실제 실행 결과만 채웁니다. HTML은 브라우저에서 열어 인쇄 → PDF 저장이 가능합니다.

정답의 크기·위치는 사후 분석에서는 알지만 놓친 새 물체에서는 알 수 없습니다. 따라서 “작은 물체는 재검사”처럼 실제로 알 수 없는 정보를 운영 규칙으로 쓰지 않습니다.

In [ ]:
def markdown_table(df):
    d=df.copy()
    for c in d.select_dtypes(include='float').columns:d[c]=d[c].map(lambda x:f'{x:.4f}' if pd.notna(x) else '평가 불가')
    return '| '+' | '.join(map(str,d.columns))+' |\n|'+'|'.join(['---']*len(d.columns))+'|\n'+'\n'.join('| '+' | '.join(map(str,row))+' |' for row in d.itertuples(index=False,name=None))
def write_report(work,context):
    out=work/'submission';out.mkdir(exist_ok=True)
    sections=[]
    def paragraph(title,text):sections.append(('text',title,text))
    names={'role':'자료 역할','images':'영상 수','objects':'정답 수','dates':'날짜 수','machine':'장비','n_objects':'영상당 정답 수',
        'candidate':'모델 조건','seed':'시드','fold':'평가 묶음','mean_f2':'평균 F2','mean_precision':'평균 정밀도',
        'worst_recall':'최저 재현율','recall_std':'재현율 표준편차','goal_fraction':'목표 충족 비율','runs':'반복 수',
        'threshold':'점수 기준','iou':'IoU 기준','tp':'맞게 탐지','fp':'잘못된 예측','fn':'놓친 정답','precision':'정밀도',
        'recall':'재현율','f1':'F1','f2':'F2','fp_per_image':'영상당 잘못된 예측','all_found_images':'모든 정답 탐지 영상',
        'normal_images':'확인된 정상 영상','normal_false_alarm_rate':'정상 오경보율','target_met':'목표 달성',
        'evaluation':'평가 대상','condition':'조건','group':'구간','detected':'탐지 수','missed':'미탐지 수','cutoff':'학습 경계값',
        'method':'제거 방법','dilation':'마스크 확장 크기','trials':'대조시험 수','new_alerts':'새 경보 수','new_alert_rate':'새 경보 비율','note':'해석'}
    terms={'final_fit':'최종 학습','final_calibration':'최종 보정','historical_seen_test':'기존 시험 재평가',
        'min_side_px':'박스 짧은 변(픽셀)','contrast_cnr':'주변 대비','edge_distance_px':'추정 가장자리 거리',
        'aspect_ratio':'박스 가로세로 비율','inpaint_overlap':'정답 내 제거 영역 비율','machine':'장비','count_bin':'영상 내 이물질 개수',
        'train_Q25_or_less':'학습 하위 25% 경계 이하','above_train_Q25':'학습 하위 25% 경계 초과',
        'elongated_1.5_plus':'긴 변/짧은 변 1.5 이상','below_1.5':'긴 변/짧은 변 1.5 미만',
        'above_25pct':'25% 초과','up_to_25pct':'25% 이하','single':'1개','multi':'여러 개','unknown':'계산 불가'}
    def tab(title,df):
        readable=df.copy()
        for column in readable.columns:
            if readable[column].dtype==object:readable[column]=readable[column].map(lambda x:terms.get(x,x) if isinstance(x,str) else x)
        readable=readable.rename(columns=names)
        sections.append(('table',title,readable))
    def figure(title,path):
        if path.exists():sections.append(('image',title,path))
    status='SMOKE 실행 확인용 - 제출 금지' if MODE=='smoke' else '제출용 분석 결과 - 현장 안전 인증 아님'
    paragraph('0. 결과의 범위',f'{status}. 실행 계약 {work.name}. 사용자가 제공한 ④ 출제문과 색상 표시 추가 안내를 기준으로 작성했다. 공식 공고의 별도 배점/양식은 자동 접근 실패로 확인되지 않았다. 보고서의 수치는 실행 결과에서 생성되며 기존 98.67%를 새 결과로 재사용하지 않는다.')
    audit=context['audit']
    paragraph('1. 문제와 데이터',f"목표는 이물질 위치 탐지와 미탐지 조건 분석, 점수 기준과 재검사 제안이다. 주 TXT {audit['main_txt']}개 중 판본 충돌 {audit['quarantined_images']}장은 보류했다. 사용 영상 {audit['eligible_images']}장, 정답 {audit['objects']}개, 확인된 정상 영상 {audit['confirmed_negatives']}장이다. 라벨 없는 BMP를 정상으로 간주하지 않았고 이미지 표시와 TXT가 다르면 TXT만 정답으로 사용했다. 별도 실습 TXT는 주 분석 범위 밖으로 기록했다.")
    tab('최종 자료 역할',context['role_summary'])
    tab('장비·개수별 자료 분포',context['support'])
    paragraph('2. 전처리와 EDA','RGB 채널 차이 20 초과 픽셀을 찾아 3×3 확장 후 반경 3 인페인팅을 적용했다. TXT 좌표는 제거 함수에 넣지 않았다. Telea/NS 두 방법을 비교하고 모델 입력은 표시 제거된 흑백 PNG만 사용했다. 색상 마스크 자체를 입력하지 않는다. 복원 원본이 없어 메운 흔적의 영향을 완전히 배제할 수는 없다. 크기는 픽셀이고 mm 크기·소재·밀도는 확인되지 않았다. 대비는 주위 밝기와의 차이이며 물리적 밀도로 해석하지 않는다. 가장자리는 밝기로 추정한 제품 영역, 형상은 박스 종횡비 대용 지표이다.')
    figure('학습 자료의 조건 분포',work/'figures'/'eda.png')
    figure('원본과 표시 제거 비교',work/'figures'/'preprocessing.png')
    paragraph('3. 평가 설계','기존에 확인한 2020-09-05 이후 75장은 historical_seen_test로 고정해 모델 선택에 사용하지 않았다. 개발 자료에서 날짜 묶음 단위로 최종 보정 자료를 별도 확보하고, 나머지에서 3개 바깥 평가 묶음을 만들었다. 각 바깥 반복의 학습 부분을 다시 내부 학습/검증으로 나눈다. 내부 검증만으로 조기 종료·조건 선택·점수 기준을 정하고 해당 바깥 평가에서는 변경하지 않는다. 장비와 단일/다중 개수 분포를 고려하는 분리 탐색에는 모델 점수를 사용하지 않았다. 기존 프로젝트를 이미 살펴본 상태에서 설계를 보완했으므로 새로운 독립 외부 검증을 대체하지 않는다. 날짜가 달라도 같은 실제 제품일 가능성은 남는다.')
    paragraph('학습량과 재현','정식 실행은 4조건 × 3시드 × 3바깥반복 + 최종 3시드 = 39회 학습이다. 각 학습은 최대 120 epoch, 조기 종료 20회다. 사전학습 YOLO11n/11s, 640/960 입력, Telea/NS를 비교한다. 학습 완료 기록·원본/라벨/가중치 해시·설정 계약을 저장한다. 완료된 학습은 재사용하고 미완료 last.pt가 복구 가능할 때 이어서 학습한다. 최종 대표 시드는 성능을 본 뒤 고르지 않고 42로 고정했다.')
    tab('내부 검증을 이용한 후보 선택',context['selection_summary'])
    tab('바깥 평가: 각 시드에서 별도로 집계',context['cv_seed_summary'])
    tab('바깥 평가: 묶음·시드별 결과',context['cv_metrics'])
    figure('반복 평가의 변동',work/'figures'/'cv_recall.png')
    paragraph('4. 최종 모델과 판정 기준',f"선택 조건 {context['selected']}, 대표 시드 42. 최종 학습/보정 자료로 새 모델을 학습하고 경보 기준을 정했다. 목표 재현율 {CONFIG['recall_target']:.0%}는 실험자가 정한 경험적 조건이며 대회 지정 안전 수치가 아니다. 탐지 점수는 보정된 불량 확률이 아니다. 기준은 IoU 0.5의 일대일 매칭이며 중복 예측은 추가 정답으로 세지 않는다. 목표를 못 채우면 target_met=False로 기록한다.")
    tab('최종 시드별 보정 결과',context['final_cal_metrics'])
    tab('기존 시험 재평가: 새로운 맹검 시험이 아님',context['historical_metrics'])
    representative=context['cv_seed_summary'].query('seed == 42').iloc[0]
    historical=context['historical_metrics'].query('seed == 42').iloc[0]
    paragraph('성적표를 쉽게 읽으면',f"대표 시드의 바깥 평가에서는 정답 {int(representative.objects)}개 중 {int(representative.tp)}개를 위치까지 맞게 찾고 {int(representative.fn)}개를 놓쳤다. 정답과 맞지 않는 예측은 {int(representative.fp)}개다. 재현율은 {representative.recall:.1%}, 정밀도는 {representative.precision:.1%}다. 이는 각 반복에서 내부 검증으로 선택한 모델의 평가를 모은 값이다. 최종 단일 모델의 기존 시험 재평가는 정답 {int(historical.objects)}개 중 {int(historical.tp)}개 탐지, {int(historical.fn)}개 미탐지다. 두 표는 자료와 평가 절차가 다르므로 숫자만으로 개선 폭을 계산하지 않는다. mAP50은 위치 겹침 기준 0.5에서의 종합 점수, mAP50–95는 더 엄격한 여러 위치 기준의 평균이다.")
    if context.get('external_metrics') is not None:tab('사용자 제공 외부 평가',pd.DataFrame([context['external_metrics']]))
    else:paragraph('외부 평가 상태','새로운 외부 자료는 설정되지 않아 실행하지 않았다. 정상 오경보율과 실제 출하 안전성을 검증했다고 주장하지 않는다. 외부 자료를 추가할 경우 모델/기준을 고정한 뒤 별도 경로에서 평가하며, 명시적 빈 TXT가 있는 정상 영상만 정상으로 센다.')
    tab('위치만 외우는 단순 기준과의 비교',context['baseline'])
    paragraph('5. 미탐지 조건','크기·대비·추정 경계 거리의 경계는 각 평가 모델의 학습 자료 하위 25%로 정한다. TXT로 계산한 사후 조건을 운영 때 놓친 이물질의 위치를 아는 것처럼 사용하지 않는다. 같은 실패가 여러 조건에 중복 포함될 수 있고 표본이 적은 조건은 원인으로 단정할 수 없다. 근처에 박스가 있으나 IoU 기준 미달인 경우와 대응 탐지가 없는 경우를 구분했다. 반복 시드의 정답을 합쳐 독립 표본 수를 부풀리지 않는다.')
    tab('바깥 평가 조건별 결과: 대표 시드 42',context['cv_conditions'])
    tab('기존 시험 조건별 결과: 대표 시드 42',context['historical_conditions'])
    sufficient=context['cv_conditions'].query('objects >= 10').sort_values(['recall','objects'],ascending=[True,False]).head(3)
    if len(sufficient):
        findings='; '.join(f"{terms.get(r.condition,r.condition)} / {terms.get(r.group,r.group)}: {int(r.objects)}개 중 {int(r.missed)}개 미탐지, 재현율 {r.recall:.1%}" for r in sufficient.itertuples())
        paragraph('우선 확인할 조건',f'바깥 평가에서 표본 10개 이상인 조건 중 재현율이 낮은 순서는 다음과 같다. {findings}. 같은 물체가 여러 조건에 포함될 수 있다. 차이가 물체 자체 때문인지 날짜·장비·표시 제거 때문인지는 이 비교만으로 단정할 수 없다. 현장에서는 해당 조건을 포함한 독립 자료를 더 확보해 확인해야 한다.')
    figure('기존 시험의 어려운 사례',work/'figures'/'hard_cases.png')
    paragraph('6. 표시 제거 영향 점검','같은 보정 영상에 제거 방법 변경·마스크 확장을 적용해 점수 민감도를 확인했다. 정답과 겹치지 않는 제품 내부에 12픽셀짜리 가짜 색 사각형을 추가한 뒤 다시 지우는 대조시험도 했다. 제거 흔적 근처에 새로운 경보가 생기는지 검사한다. 이 시험은 실제 정상 제품 오경보율도, 표시 없는 원본의 복원 성공률도 아니다. 대조 위치는 분석을 위해 TXT를 참고하지만 모델에 넣는 제거 함수는 TXT를 받지 않는다.')
    tab('전처리 민감도',pd.read_csv(work/'preprocessing_stress.csv'))
    tab('가짜 표시 제거 대조시험',pd.DataFrame([context['stress']]))
    figure('대조시험의 위치 예시',work/'figures'/'artifact_control.png')
    policy=context['policy']
    paragraph('7. 판정과 재검사 제안',f"높은 기준 {policy['high']:.3f} 이상은 이물질 의심으로 격리 후 확인한다. 낮은 기준 {policy['low']:.3f} 이상부터 높은 기준 미만은 재검사한다. 기준이 같으면 중간 구간이 없다. 낮은 기준 미만이거나 탐지가 없어도 자동 정상 통과하지 않고 기존 검사/재검사를 유지한다. 영상 품질 불량·미등록 장비/제품은 점수와 무관하게 확인한다. 놓친 물체의 실제 크기/위치는 운영 시 알 수 없으므로 사후 미탐지 표를 곧바로 실시간 경보 규칙으로 바꾸지 않는다. 재검사는 다른 촬영 조건 또는 기존 검사/사람 확인을 뜻하며 같은 영상을 반복 예측하는 것만으로 독립 검사가 되지 않는다.")
    paragraph('8. 한계와 제출 전 확인','확인된 정상 자료 부족, 미확정 TXT 3장, 표시 없는 원본 부재, 물리적 밀도/소재 정보 부재, 실제 제품 ID 부재는 코드만으로 해결하지 못했다. 역사적 시험은 이미 확인한 자료이다. 외부 자료에 대한 불량/정상 비율, 장비, 제품, 재검사 부담과 실제 안전 목표를 확정해야 한다. 모델 성능이 좋더라도 출하 승인/안전 인증으로 사용하지 않는다. 제출 양식과 배점에 맞는 최종 편집 및 TXT 판본 확인은 별도 필요하다.')
    paragraph('참고','제공된 ④ 출제문과 색상 표시 추가 안내. https://www.kamp-ai.kr/contestNoticeDetail?CPT_NOTICE_SEQ=28 ; https://scikit-learn.org/stable/modules/cross_validation.html ; https://docs.ultralytics.com/modes/train/ ; https://docs.opencv.org/4.x/df/d3d/tutorial_py_inpainting.html')
    markdown=['# X-ray 이물질 탐지 및 미탐지 조건 분석\n'];body=[]
    for kind,title,value in sections:
        markdown.append('## '+title+'\n');body.append('<section><h2>'+html.escape(title)+'</h2>')
        if kind=='text':markdown.append(value+'\n');body.append('<p>'+html.escape(value)+'</p>')
        elif kind=='table':
            markdown.append(markdown_table(value)+'\n');body.append(value.to_html(index=False,escape=True,float_format=lambda x:f'{x:.4f}'))
        else:
            rel='../figures/'+value.name;markdown.append(f'![{title}]({rel})\n')
            body.append('<img src="data:image/png;base64,'+base64.b64encode(value.read_bytes()).decode()+'"/>')
        body.append('</section>')
    (out/'제출보고서.md').write_text('\n'.join(markdown),encoding='utf-8')
    style='body{font-family:"Malgun Gothic",sans-serif;max-width:1050px;margin:35px auto;color:#203143;line-height:1.75}h1,h2{color:#163650}table{border-collapse:collapse;width:100%;font-size:12px;margin:15px 0}th{background:#17354d;color:white}td,th{border:1px solid #ccd6dd;padding:7px;text-align:left}tr:nth-child(even){background:#f1f6f8}img{max-width:100%}p{white-space:pre-line}.status{padding:16px;background:#fff0ce;font-weight:bold}@media print{@page{size:A4;margin:16mm}body{font-size:10pt}h2{break-after:avoid}tr,img{break-inside:avoid}thead{display:table-header-group}section{margin-bottom:20px}}'
    (out/'제출보고서.html').write_text('<!doctype html><html lang="ko"><meta charset="utf-8"><title>X-ray 제출 분석</title><style>'+style+'</style><body><h1>X-ray 이물질 탐지 및 미탐지 조건 분석</h1><div class="status">'+html.escape(status)+'</div>'+''.join(body)+'</body></html>',encoding='utf-8')
    return out

## 13. 최종 모델과 제출 자료 묶기
최종 대표 모델, TXT 없이 새 BMP를 읽는 추론 코드, 학습 노트북, 환경 목록, 보고서, 근거 표를 저장합니다. `predict.py`는 전처리와 탐지를 한 번에 수행합니다. 탐지가 없어도 자동 정상으로 판정하지 않습니다.

In [ ]:
def export_package(work,record,policy,report_dir):
    # 추론 코드의 전처리도 노트북 함수와 같게 내보냅니다.
    infer_source=r'''from pathlib import Path
import argparse,json,os
import numpy as np
from PIL import Image,ImageDraw
import cv2
HERE=Path(__file__).resolve().parent
os.environ.setdefault('YOLO_CONFIG_DIR',str(HERE))
from ultralytics import YOLO
def main():
    parser=argparse.ArgumentParser();parser.add_argument('--image',required=True);parser.add_argument('--output',default='prediction.png');args=parser.parse_args()
    cfg=json.loads((HERE/'inference_config.json').read_text(encoding='utf-8'))
    rgb=np.asarray(Image.open(args.image).convert('RGB'))
    mask=(np.ptp(rgb.astype(np.int16),axis=2)>20).astype(np.uint8)*255
    mask=cv2.dilate(mask,np.ones((3,3),np.uint8));gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)
    if mask.any():gray=cv2.inpaint(gray,mask,3,cv2.INPAINT_TELEA if cfg['method']=='telea' else cv2.INPAINT_NS)
    model=YOLO(str(HERE/'best.pt'));p=model.predict(cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR),imgsz=cfg['imgsz'],conf=cfg['low'],iou=.5,max_det=100,verbose=False)[0]
    boxes=np.column_stack((p.boxes.xyxy.cpu().numpy(),p.boxes.conf.cpu().numpy()))
    decision='검출 없음: 정상 확정 불가, 기존 검사/재검사 유지'
    if len(boxes):decision='이물질 의심: 격리 후 확인' if boxes[:,4].max()>=cfg['high'] else '의심 영역: 재검사'
    im=Image.fromarray(gray).convert('RGB');draw=ImageDraw.Draw(im)
    for b in boxes:draw.rectangle(b[:4].tolist(),outline='red',width=2);draw.text((b[0],max(0,b[1]-12)),f'{b[4]:.3f}',fill='red')
    im.save(args.output)
    Path(args.output).with_suffix('.json').write_text(json.dumps({'decision':decision,'boxes_xyxy_score':boxes.tolist(),'automatic_release':False},ensure_ascii=False,indent=2),encoding='utf-8')
    print(decision)
if __name__=='__main__':main()
'''
    (report_dir/'predict.py').write_text(infer_source,encoding='utf-8')
    shutil.copyfile(record['checkpoint'],report_dir/'best.pt')
    dump(report_dir/'inference_config.json',dict(method=record['candidate']['method'],imgsz=record['candidate']['imgsz'],**policy))
    notebook=load(ROOT/'baseline.ipynb')
    for cell in notebook['cells']:
        if cell['cell_type']=='code':cell.update(outputs=[],execution_count=None)
    dump(report_dir/'baseline.ipynb',notebook)
    shutil.copyfile(ROOT/'requirements.txt',report_dir/'requirements.txt')
    readme='이 묶음은 분석 결과와 추론용 모델입니다. 원본 데이터는 포함하지 않습니다.\n환경: requirements.txt 설치 후 python predict.py --image 원본.bmp --output prediction.png\nTXT 없이 추론하며 자동 정상 통과는 하지 않습니다. 보고서 HTML을 브라우저에서 열어 인쇄/PDF 저장할 수 있습니다.\n노트북 재학습은 baseline.ipynb와 원본 데이터 폴더를 같은 위치에 두고 실행하세요.\n'
    (report_dir/'README.txt').write_text(readme,encoding='utf-8')
    if MODE=='smoke':return None
    package=work/'Xray_제출자료.zip'
    with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
        for p in report_dir.rglob('*'):
            if p.is_file():z.write(p,'submission/'+p.relative_to(report_dir).as_posix())
        for p in (work/'figures').glob('*.png'):z.write(p,'figures/'+p.name)
        for p in work.glob('*.csv'):z.write(p,'evidence/'+p.name)
        for p in work.glob('*.json'):z.write(p,'evidence/'+p.name)
        for run in (work/'runs').iterdir():
            if run.is_dir():
                for name in ('results.csv','args.yaml','requested.json','completed.json'):
                    p=run/name
                    if p.is_file():z.write(p,'evidence/training/'+run.name+'/'+name)
    return package

## 14. 전체 실행 순서 연결하기
위 기능을 순서대로 연결합니다. 후보는 내부 검증 결과로만 선택합니다. 기존 시험의 결과는 보고서 작성에 쓰지만 모델 선정에는 쓰지 않습니다.

시드별 결과를 따로 제시해 같은 정답을 여러 번 본 것을 새로운 표본처럼 부풀리지 않습니다. 가장 어려운 실패 사례도 그림으로 저장합니다.

In [ ]:
def main():
    protocol_tests();manifest,audit=inventory()
    source=json.loads((ROOT/'baseline.ipynb').read_text(encoding='utf-8'))
    code_text='\n'.join(''.join(c['source']) if isinstance(c['source'],list) else c['source'] for c in source['cells'] if c['cell_type']=='code')
    contract={'config':CONFIG,'source_hash':digest(code_text.encode()),'manifest_hash':digest(manifest.to_csv(index=False).encode()),
        'packages':{p:package_metadata.version(p) for p in ['torch','torchvision','ultralytics','numpy','opencv-python','scikit-learn']}}
    run_id=MODE+'_'+digest(json.dumps(contract,sort_keys=True).encode())[:12]
    work=ROOT/'submission_work'/run_id;work.mkdir(parents=True,exist_ok=True);(work/'figures').mkdir(exist_ok=True)
    contract['id']=run_id;dump(work/'contract.json',contract);dump(work/'data_audit.json',audit)
    save_csv(manifest,work/'manifest.csv');print('실험 폴더:',work,flush=True)
    core,cal,historical,plans=plan_splits(manifest,work)
    truth,features=prepare(manifest,work)
    roles=pd.read_csv(work/'final_split_assignments.csv',dtype={'date':str,'machine':str})
    role_summary=roles.groupby('role').agg(images=('image_id','size'),objects=('n_objects','sum'),dates=('date','nunique')).reset_index()
    support=roles.groupby(['role','machine','n_objects']).size().rename('images').reset_index()
    display(role_summary);display(support)
    # 그림과 조건 경계는 학습 부분을 중심으로 읽습니다. 시험 조건은 최종 분석용입니다.
    train_features=features[features.image_id.isin(core.image_id)]
    fig,ax=plt.subplots(1,3,figsize=(11,3))
    for a,c,title in zip(ax,['min_side_px','contrast_cnr','edge_distance_px'],['크기(픽셀)','주변 대비','추정 경계 거리']):a.hist(train_features[c].dropna(),bins=20);a.set_title(title)
    fig.tight_layout();fig.savefig(work/'figures'/'eda.png',dpi=150);plt.close(fig)
    example=core.iloc[0];raw=np.asarray(Image.open(ROOT/example.raw).convert('RGB'));clean,_=clean_rgb(raw)
    fig,ax=plt.subplots(1,2,figsize=(9,4))
    for a,im,title in zip(ax,[raw,clean],['원본: 색상 표시 포함','모델 입력: 표시 제거']):a.imshow(im,cmap='gray');a.set_title(title);a.axis('off')
    fig.tight_layout();fig.savefig(work/'figures'/'preprocessing.png',dpi=150);plt.close(fig)
    all_inner=[];outer_results=[];outer_images=[];outer_conditions=[];baseline_rows=[];fit_records=[]
    for fold,(fit,tune,evaluation) in enumerate(plans):
        inner=[];models={};policies={}
        for candidate in CONFIG['candidates']:
            for seed in CONFIG['seeds']:
                tag=f'fold{fold}_{candidate["name"]}_seed{seed}'
                record=fit_model(work,tag,candidate,seed,fit,tune);fit_records.append(record)
                pred=predict(work,record,tune,tag+'_inner')
                policy,grid=threshold_policy(pred,tune,truth);save_csv(grid,work/(tag+'_thresholds.csv'))
                metrics=score(pred,tune,truth,policy['low'])[0]
                row=dict(fold=fold,candidate=candidate['name'],seed=seed,target_met=policy['target_met'],**metrics)
                inner.append(row);all_inner.append(row);models[(candidate['name'],seed)]=record;policies[(candidate['name'],seed)]=policy
        chosen,summary=choose_candidate(inner);save_csv(summary,work/f'fold{fold}_inner_selection.csv')
        dump(work/f'fold{fold}_frozen_before_outer.json',{'candidate':chosen,'policies':{str(s):policies[(chosen,s)] for s in CONFIG['seeds']},
            'fit_ids':fit.image_id.tolist(),'tune_ids':tune.image_id.tolist(),'evaluation_ids':evaluation.image_id.tolist()})
        for seed in CONFIG['seeds']:
            record=models[(chosen,seed)];policy=policies[(chosen,seed)]
            pred=predict(work,record,evaluation,f'fold{fold}_seed{seed}_outer')
            metrics,images,obj=score(pred,evaluation,truth,policy['low'])
            outer_results.append(dict(fold=fold,seed=seed,candidate=chosen,**metrics))
            images['fold']=fold;images['seed']=seed;outer_images.append(images)
            if seed==42:
                conditions_table,_=conditions(obj,features,fit.image_id);conditions_table['fold']=fold;outer_conditions.append(conditions_table)
                save_csv(obj,work/f'fold{fold}_outer_objects_seed42.csv')
        b=position_baseline(fit,evaluation,truth)
        baseline_rows.append(dict(evaluation=f'outer_fold_{fold}',**score(b,evaluation,truth,.5)[0]))
        save_csv(pd.DataFrame(all_inner),work/'inner_metrics.csv');save_csv(pd.DataFrame(outer_results),work/'outer_metrics.csv')
    outer_frame=pd.DataFrame(outer_results);outer_image_frame=pd.concat(outer_images,ignore_index=True)
    save_csv(outer_image_frame,work/'outer_image_results.csv')
    seed_summary=[]
    for seed,d in outer_image_frame.groupby('seed'):
        assert not d.image_id.duplicated().any()
        tp,fp,fn=map(int,d[['tp','fp','fn']].sum())
        seed_summary.append(dict(seed=seed,images=len(d),objects=tp+fn,tp=tp,fp=fp,fn=fn,recall=tp/max(tp+fn,1),precision=tp/max(tp+fp,1),f1=2*tp/max(2*tp+fp+fn,1)))
    seed_summary=pd.DataFrame(seed_summary);save_csv(seed_summary,work/'outer_seed_summary.csv')
    dump(work/'outer_seed42_interval.json',cluster_interval(outer_image_frame[outer_image_frame.seed==42]))
    cc=pd.concat(outer_conditions,ignore_index=True)
    cv_conditions=cc.groupby(['condition','group']).agg(objects=('objects','sum'),detected=('detected','sum'),missed=('missed','sum')).reset_index()
    cv_conditions['recall']=cv_conditions.detected/cv_conditions.objects.replace(0,np.nan)
    save_csv(cc,work/'outer_condition_by_fold.csv');save_csv(cv_conditions,work/'outer_conditions_seed42.csv')
    selected,summary=choose_candidate(all_inner);save_csv(summary,work/'final_candidate_selection_inner_only.csv')
    candidate=next(c for c in CONFIG['candidates'] if c['name']==selected)
    final_records={};final_policies={};cal_results=[]
    for seed in CONFIG['seeds']:
        tag=f'final_{selected}_seed{seed}';record=fit_model(work,tag,candidate,seed,core,cal,historical);fit_records.append(record)
        pred=predict(work,record,cal,tag+'_calibration');policy,grid=threshold_policy(pred,cal,truth)
        save_csv(grid,work/f'final_seed{seed}_thresholds.csv')
        cal_results.append(dict(seed=seed,**score(pred,cal,truth,policy['low'])[0],target_met=policy['target_met']))
        final_records[seed]=record;final_policies[seed]=policy
    save_csv(pd.DataFrame(cal_results),work/'final_calibration_metrics.csv')
    frozen={'candidate':selected,'representative_seed':42,'models':final_records,'policies':final_policies,
            'selection':'inner validation only; final thresholds on separate calibration; no historical test selection',
            'frozen_at':datetime.now().astimezone().isoformat()}
    dump(work/'frozen_before_historical_evaluation.json',frozen)
    hist_results=[];primary_objects=None;primary_pred=None
    for seed in CONFIG['seeds']:
        record=final_records[seed];policy=final_policies[seed]
        pred=predict(work,record,historical,f'final_seed{seed}_historical')
        metrics,images,obj=score(pred,historical,truth,policy['low']);hist_results.append(dict(seed=seed,**metrics))
        save_csv(images,work/f'historical_images_seed{seed}.csv');save_csv(obj,work/f'historical_objects_seed{seed}.csv')
        if seed==42:
            primary_objects=obj;primary_pred=pred
            dump(work/'historical_seed42_interval.json',cluster_interval(images))
    hist_results=pd.DataFrame(hist_results);save_csv(hist_results,work/'historical_metrics.csv')
    hist_conditions,merged=conditions(primary_objects,features,core.image_id)
    save_csv(hist_conditions,work/'historical_conditions.csv');save_csv(merged,work/'historical_object_details.csv')
    baseline_rows.append(dict(evaluation='historical_seen_test',**score(position_baseline(core,historical,truth),historical,truth,.5)[0]))
    save_csv(pd.DataFrame(baseline_rows),work/'position_only_baseline.csv')
    primary=final_records[42];policy=final_policies[42]
    # 표준 위치 검출 지표를 계산하며 모델과 판정 기준은 바꾸지 않습니다.
    model=YOLO(primary['checkpoint'])
    metrics=model.val(data=primary['data_yaml'],split='test',imgsz=candidate['imgsz'],batch=candidate['batch'],
        device=DEVICE,workers=0,conf=CONFIG['prediction_floor'],iou=CONFIG['nms_iou'],max_det=100,plots=True,
        project=str(work/'runs'),name='standard_historical_metrics',exist_ok=True,verbose=False)
    standard={'mAP50':float(metrics.box.map50),'mAP50_95':float(metrics.box.map),'cohort':'historical_seen_test'}
    dump(work/'standard_detection_metrics.json',standard);del model
    hist_results['mAP50']=np.where(hist_results.seed==42,standard['mAP50'],np.nan)
    hist_results['mAP50_95']=np.where(hist_results.seed==42,standard['mAP50_95'],np.nan)
    save_csv(hist_results,work/'historical_metrics.csv')
    # 가장 어려운 사례를 자동 저장합니다. 주황=예측, 초록=TXT.
    ids=merged.sort_values(['detected','best_iou']).image_id.drop_duplicates().head(6).tolist()
    fig,axes=plt.subplots(2,3,figsize=(12,8))
    for ax,image_id in zip(axes.flat,ids):
        im=Image.open(work/'prepared'/candidate['method']/'images'/(image_id+'.png')).convert('RGB');d=ImageDraw.Draw(im)
        for b in truth[image_id]:d.rectangle(b.tolist(),outline='green',width=1)
        for b in primary_pred[image_id]:
            if b[4]>=policy['low']:d.rectangle(b[:4],outline='orange',width=1)
        ax.imshow(im);ax.set_title(image_id,fontsize=8);ax.axis('off')
    for ax in axes.flat[len(ids):]:ax.axis('off')
    fig.suptitle('초록=TXT 정답 / 주황=예측');fig.tight_layout();fig.savefig(work/'figures'/'hard_cases.png',dpi=150);plt.close(fig)
    fig,ax=plt.subplots(figsize=(7,4))
    for seed,d in outer_frame.groupby('seed'):ax.plot(d.fold,d.recall,marker='o',label=f'seed {seed}')
    ax.set(xlabel='바깥 평가 묶음',ylabel='재현율',ylim=(0,1.02));ax.legend();fig.tight_layout();fig.savefig(work/'figures'/'cv_recall.png',dpi=150);plt.close(fig)
    stress=stress_tests(work,primary,policy,cal,truth)
    ext=external_data(work,manifest,truth);ext_metrics=None
    if ext is not None:
        pred=predict(work,primary,ext,'external_fixed_model');ext_metrics,ei,eo=score(pred,ext,truth,policy['low'])
        dump(work/'external_metrics.json',ext_metrics);save_csv(ei,work/'external_images.csv');save_csv(eo,work/'external_objects.csv')
    expected=CONFIG['outer_folds']*len(CONFIG['candidates'])*len(CONFIG['seeds'])+len(CONFIG['seeds'])
    assert len(fit_records)==expected
    save_csv(pd.DataFrame([{k:r[k] for k in ['tag','seed','epochs_run','best_epoch','training_log_seconds','changed_shared_weight_tensors','checkpoint_sha256']} for r in fit_records]),work/'training_audit.csv')
    context=dict(audit=audit,role_summary=role_summary,support=support,selection_summary=summary,cv_seed_summary=seed_summary,
        cv_metrics=outer_frame,selected=selected,final_cal_metrics=pd.DataFrame(cal_results),historical_metrics=hist_results,
        baseline=pd.DataFrame(baseline_rows),cv_conditions=cv_conditions,historical_conditions=hist_conditions,
        policy=policy,stress=stress,external_metrics=ext_metrics)
    # 제출 묶음을 만들기 전에 원본이 바뀌지 않았는지 확인합니다.
    assert all(file_hash(ROOT/r.raw)==r.raw_sha256 and file_hash(ROOT/r.label)==r.label_sha256 for r in manifest.itertuples())
    report_dir=write_report(work,context);package=export_package(work,primary,policy,report_dir)
    dump(work/'COMPLETE.json',{'mode':MODE,'completed_at':datetime.now().astimezone().isoformat(),'trainings':expected,
        'report_dir':str(report_dir),'package':str(package) if package else None,'external_evaluated':ext is not None,
        'field_safety_validated':False,'code_source_hash':contract['source_hash']})
    if package:
        with zipfile.ZipFile(package,'a',zipfile.ZIP_DEFLATED) as z:z.write(work/'COMPLETE.json','evidence/COMPLETE.json')
    if MODE=='full':dump(ROOT/'LATEST_SUBMISSION.json',load(work/'COMPLETE.json'))
    else:dump(ROOT/'SMOKE_CHECK.json',load(work/'COMPLETE.json'))
    display(seed_summary);display(hist_results)
    print('완료:',report_dir,'\n제출 묶음:',package or 'smoke 모드는 제출 ZIP을 만들지 않음',flush=True)
    return work

## 15. 전체 실행 시작
아래 셀은 오래 걸릴 수 있습니다. `학습: fold...`가 나오면 실제 학습이 진행 중입니다. 모든 단계가 끝나야 `완료`와 제출 폴더 경로가 표시됩니다.

정식 실행에서 만들어진 `LATEST_SUBMISSION.json`과 `COMPLETE.json`을 확인하세요. 중간 파일만 생겼다고 전체가 완료된 것은 아닙니다. 실행 후 보고서의 목표 재현율 달성 여부, 조건별 미탐지, 표시 제거 영향과 한계를 반드시 읽어보세요.

In [ ]:
RESULT_DIR = main()